# Hierarchical control of visual perceptual learning with model-synthesized stimuli

Code accompanying Valibeigi, Pandey, Pack & Bakhtiari.

This notebook reproduces the computational parts of the paper:

| Section | What it does | Paper |
|---|---|---|
| **1. Setup** | Clones DorsalNet, downloads its checkpoints, installs dependencies, loads the frozen model and attaches forward hooks | 2.2.1 |
| **2. Stimulus generation** | PCA readout directions, synthesis of res0- and res2-targeted stimulus pairs, graded optimization levels, GIF export | 2.2 |
| **3. ANN readout modeling** | Linear readouts of the frozen DorsalNet trained on the synthesized stimuli and tested for location and stimulus-category transfer | 2.4 |
| **4. Visualization** | Synthesized stimuli, training/testing performance and specificity indices | Figs 2, 6, 7 |

**Before you start**
* Run the notebook from the repository root.
* A CUDA GPU is strongly recommended (stimulus synthesis and the 10 000-epoch readout training are slow on CPU).
* Reproducibility: the global seed is set to **42** before stimulus generation (Section 2.3) and again before the ANN readout experiment (Section 3). Keep the layer order in `Model_Variables.py` and the readout order in `beh_Model_Variables.py` unchanged; the random-number stream carries over from one layer/readout to the next.

---
# 1. Setup

DorsalNet (Mineault et al., 2021) is a 3D residual network trained to estimate self-motion from egocentric videos. Its stages `s1`, `res0`–`res3` align with areas V1, MT and MST of the primate dorsal stream. The model code lives in the [`your-head-is-there-to-move-you-around`](https://github.com/patrickmineault/your-head-is-there-to-move-you-around) repository (cloned below as `yhit/`), which also ships the `lucentpatch` package used here.

Steps 1.2–1.5 only need to be run once.

### 1.1 Paths

In [ ]:
import os
import sys

# Run this notebook from the repository root
REPO_ROOT = os.path.abspath('.')

YHIT_DIR        = os.path.join(REPO_ROOT, 'yhit')                 # DorsalNet code (+ lucentpatch)
CHECKPOINT_DIR  = os.path.join(REPO_ROOT, 'checkpoints')          # DorsalNet weights
LUCENT_DIR      = os.path.join(REPO_ROOT, 'lucent')               # lucent library (cloned + installed)
PCA_DIR         = os.path.join(REPO_ROOT, 'pca_components')       # PCA readout directions (Section 2.1)
STIM_GEN_DIR    = os.path.join(REPO_ROOT, 'stimulus_generation')  # Section 2 code
ANN_READOUT_DIR = os.path.join(REPO_ROOT, 'ann_readout')          # Section 3 code
RESULTS_DIR     = os.path.join(REPO_ROOT, 'results')              # ANN results
FIGURE_DIR      = os.path.join(RESULTS_DIR, 'figures')            # Section 4 figures

os.makedirs(FIGURE_DIR, exist_ok=True)

### 1.2 Clone DorsalNet (`yhit`)

In [ ]:
# https://github.com/patrickmineault/stim-gen/blob/main/notebooks/Create%20ensemble%20stimuli.ipynb
if not os.path.isdir(YHIT_DIR):
    !git clone https://github.com/patrickmineault/your-head-is-there-to-move-you-around "{YHIT_DIR}"

### 1.3 Download the DorsalNet checkpoints

In [ ]:
import gdown
import zipfile

# Google Drive file ID of the DorsalNet checkpoints
file_id = '16ABLAYyqc_fx7u6IZH0rXLnyYQbZy3HH'
url = f'https://drive.google.com/uc?id={file_id}'

save_folder = CHECKPOINT_DIR
os.makedirs(save_folder, exist_ok=True)
zip_filename = os.path.join(save_folder, 'checkpoints.zip')

if not any(f for f in os.listdir(save_folder) if f != 'checkpoints.zip'):
    # Download, unzip and remove the archive
    gdown.download(url, zip_filename, quiet=False)
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(save_folder)
    os.remove(zip_filename)
    print("Checkpoints zip file downloaded and extracted.")
else:
    print("Checkpoints already present in", save_folder)

### 1.4 Point `yhit/paths.py` to the checkpoint folder
`yhit` reads its data and checkpoint locations from `paths.py`. All of them are set to the `checkpoints/` folder.

In [ ]:
ckpt = CHECKPOINT_DIR.replace('\\', '/')  # forward slashes are safe on every OS
paths_file = os.path.join(YHIT_DIR, 'paths.py')

with open(paths_file, 'r') as file:
    lines = file.readlines()

for i in range(len(lines)):
    if lines[i].startswith('RAW_DATA ='):
        lines[i] = f'RAW_DATA = "{ckpt}/"\n'
    if lines[i].startswith('DERIVED_DATA ='):
        lines[i] = f'DERIVED_DATA = "{ckpt}/"\n'
    if lines[i].startswith('CHECKPOINTS ='):
        lines[i] = f'CHECKPOINTS = "{ckpt}/"\n'
    if lines[i].startswith('CPC_DPC ='):
        lines[i] = f'CPC_DPC = "{ckpt}/"\n'
    if lines[i].startswith('CPC_BACKBONE ='):
        lines[i] = f'CPC_BACKBONE = "{ckpt}/"\n'

with open(paths_file, 'w') as file:
    file.writelines(lines)

### 1.5 Install dependencies
1. DorsalNet requirements (conda).
2. [lucent](https://github.com/greentfrapp/lucent), installed from source.
3. Remaining packages used by this repository (`requirements.txt`).

In [ ]:
# 1. DorsalNet / yhit requirements
!conda install --file "{YHIT_DIR}/requirements.txt" -y

# 2. lucent: clone and install from source
if not os.path.isdir(LUCENT_DIR):
    !git clone https://github.com/greentfrapp/lucent.git "{LUCENT_DIR}"
!pip install "{LUCENT_DIR}"

# 3. Other packages used by this repository
!pip install -r "{REPO_ROOT}/requirements.txt"

### 1.6 Load the frozen DorsalNet and attach forward hooks
The `airsim_04` checkpoint is loaded and its weights are never updated anywhere in this notebook. Forward hooks store the post-ReLU activations of `s1` and `res0`–`res3` in the global dictionary `outputs` on every forward pass; all later code reads layer activations from this dictionary.

In [ ]:
seed = 42

In [ ]:
import numpy as np
import torch
from tqdm import tqdm
import torch.nn.functional as F
import scipy
from torch import nn
import random
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA

# ------------------------------------------------------------------
# Set the seed for everything
# ------------------------------------------------------------------
# 1. Python built-in random
random.seed(seed)
# 2. NumPy
np.random.seed(seed)
# 3. PyTorch CPU
torch.manual_seed(seed)
# 4. PyTorch GPU
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
# 5. Deterministic cuDNN behaviour
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Check if CUDA (GPU) is available
if torch.cuda.is_available():
    device = torch.device('cuda')
    print('CUDA (GPU) is available')
else:
    device = torch.device('cpu')
    print('CUDA (GPU) is not available, falling back to CPU')

%load_ext autoreload
%autoreload 2

# DorsalNet code and its paths
sys.path.append(YHIT_DIR)
from paths import *
sys.path.append(CPC_DPC)
sys.path.append(CPC_BACKBONE)
from python_dict_wrapper import wrap

import lucentpatch
import lucentpatch.transform
from lucentpatch.render import render_vis
from lucentpatch.objectives import neuron, slow, tv_slow, intensity_preservation
from lucent.optvis.objectives import channel
from lucent.optvis import param
import lucent
import lucent.optvis
import lucent.optvis.transform
from matplotlib import animation
from IPython.display import HTML

# ------------------------------------------------------------------
# Load DorsalNet (checkpoint 'airsim_04')
# ------------------------------------------------------------------
features = 'airsim_04'
args = wrap({'features': features,
             'ckpt_root': CHECKPOINTS,
             'slowfast_root': '../../slowfast',
             'ntau': 1,
             'subsample_layers': False})

from models import get_feature_model
model, hooks, data = get_feature_model(args)
model = model.to(device)

# ------------------------------------------------------------------
# Forward hooks: store each stage's post-ReLU activation in `outputs`
# ------------------------------------------------------------------
import functools
outputs = {}
def get_output(self, input, output, name=None):
    outputs[name] = output.to(device)
    return output

model.s1.relu.register_forward_hook(functools.partial(get_output, name='s1'))      # V1-like
model.res0.relu.register_forward_hook(functools.partial(get_output, name='res0'))  # early stage (stimuli: "res0")
model.res1.relu.register_forward_hook(functools.partial(get_output, name='res1'))
model.res2.relu.register_forward_hook(functools.partial(get_output, name='res2'))  # late stage (stimuli: "res2")
model.res3.relu.register_forward_hook(functools.partial(get_output, name='res3'))

---
# 2. Stimulus generation (paper Section 2.2)

For each target layer a pair of 11-frame, 164 × 164 px motion stimuli is optimized so that the two stimuli are maximally separable along a **fixed** linear readout direction of that layer, while DorsalNet stays frozen. A graded series of easier-to-harder pairs ("optimization levels") is then generated, and every stimulus is post-processed into a display-ready grayscale GIF.

Code: `stimulus_generation/`. All settings: `stimulus_generation/Model_Variables.py`.

## 2.1 PCA readout directions (optional)

The fixed readout direction of each layer is a principal component of that layer's activations to the AirSim self-motion videos used to train DorsalNet. The second component (**PC2**) gave the clearest, most coherent motion and is the one used in the paper.

The resulting files `pca_components_res0.pt` … `pca_components_res3.pt` are provided in `pca_components/`, so **this step can be skipped**. To recompute them, set `RUN_PCA = True` and `AIRSIM_DIR` to the folder that contains `output.h5`.

For each layer the cell below:
1. runs all AirSim clips through DorsalNet (batches of 4) and keeps that layer's activations;
2. computes a 20-component PCA on the flattened full activation maps and plots the scree plot and PC1 vs PC2 (exploratory; memory-intensive);
3. averages the activations over the spatial dimensions (height × width), computes a 20-component PCA on the resulting 32 channels × 11 frames = 352 features, plots it, and **saves these components** (they match the 352-dimensional classifier input).

In [ ]:
RUN_PCA = False                                      # set True to recompute the PCA components
AIRSIM_DIR = 'path/to/folder/containing/output.h5'   # only needed when RUN_PCA = True
PCA_LAYERS = ['res0', 'res1', 'res2', 'res3']

os.makedirs(PCA_DIR, exist_ok=True)

In [ ]:
import h5py

if RUN_PCA:
    # List the groups and datasets in the HDF5 file
    file_path_h5 = os.path.join(AIRSIM_DIR, 'output.h5')
    with h5py.File(file_path_h5, 'r') as h5_file:
        def list_hdf5_items(name, obj):
            if isinstance(obj, h5py.Group):
                print(f"Group: {name}")
            elif isinstance(obj, h5py.Dataset):
                print(f"Dataset: {name}")
        h5_file.visititems(list_hdf5_items)

    # Save the 'short_videos' group as a .npy file next to output.h5
    output_file_path = os.path.join(AIRSIM_DIR, 'short_video_frames.npy')
    with h5py.File(file_path_h5, 'r') as h5_file:
        short_videos_group = h5_file['short_videos']
        short_videos_data = np.array(short_videos_group)
        np.save(output_file_path, short_videos_data)

In [ ]:
if RUN_PCA:
    stimuli = np.load(output_file_path)
    stimuli = torch.from_numpy(stimuli)
    stimuli = stimuli.to(device)

    # [N, frames, C, H, W] -> [N, C, frames, H, W] (DorsalNet input layout)
    stimuli_adj = stimuli.permute(0, 2, 1, 3, 4)
    stimuli_adj_float = stimuli_adj.float()

    # Repeat the last frame so each clip has 11 frames, like the synthesized stimuli
    stimuli_adj_float = torch.cat((stimuli_adj_float, stimuli_adj_float[:, :, -1:, :, :]), dim=2).to(device)
    print(stimuli_adj_float.shape)

In [ ]:
from sklearn.decomposition import PCA

def compute_pca_components_gpu(data, num_components=20):
    '''PCA on `data` flattened to [N, features]; returns the components and
    the explained-variance ratios as tensors on data.device.'''
    # Flatten everything except the first (sample) dimension
    data_reshaped = data.view(data.size(0), -1)

    # PCA on the CPU
    data_cpu = data_reshaped.cpu().detach().numpy()
    pca = PCA(n_components=num_components)
    pca.fit_transform(data_cpu)

    pca_components = torch.tensor(pca.components_, dtype=torch.float32).to(data.device)
    explained_variance = torch.tensor(pca.explained_variance_ratio_, dtype=torch.float32).to(data.device)

    return pca_components, explained_variance

In [ ]:
if RUN_PCA:
    for layer_name_pca in PCA_LAYERS:
        print(f'===== {layer_name_pca} =====')

        # --------------------------------------------------------------
        # 1. Activations of this layer to all AirSim clips
        # --------------------------------------------------------------
        batch_size = 4
        n_samples = stimuli_adj_float.size(0)
        layer_activations = []
        with torch.no_grad():
            for start_idx in range(0, n_samples, batch_size):
                end_idx = min(start_idx + batch_size, n_samples)
                batch = stimuli_adj_float[start_idx:end_idx].to(device)
                model(batch)
                layer_activations.append(outputs[layer_name_pca].cpu())
                outputs = {}
                torch.cuda.empty_cache()  # clear CUDA cache after each batch
        layer_act = torch.cat(layer_activations, dim=0)

        # --------------------------------------------------------------
        # 2. PCA on the full activation maps (exploratory)
        # --------------------------------------------------------------
        pca_components, explained_variance = compute_pca_components_gpu(layer_act.to(device))
        print(pca_components.shape)

        plt.figure(figsize=(8, 4))
        plt.plot(range(1, len(explained_variance.cpu()) + 1), explained_variance.cpu(), marker='o')
        plt.xlabel('Principal Component')
        plt.ylabel('Explained Variance Ratio')
        save_fig_name = 'variance_' + layer_name_pca + '.png'
        plt.title(save_fig_name)
        plt.savefig(os.path.join(PCA_DIR, save_fig_name))
        plt.show()

        pca1 = pca_components[0, :].cpu()
        pca2 = pca_components[1, :].cpu()
        plt.figure(figsize=(8, 6))
        plt.scatter(pca1, pca2, alpha=0.5)
        plt.xlabel('PCA1')
        plt.ylabel('PCA2')
        plt.title('Scatter plot of PCA1 vs PCA2')
        plt.grid(True)
        plt.show()

        # --------------------------------------------------------------
        # 3. PCA on the spatially averaged activations
        #    [N, 32, 11, H, W] -> mean over H, W -> [N, 32, 11, 1, 1]
        #    -> 352 features. These components are saved and used.
        # --------------------------------------------------------------
        averaged_tensor = torch.mean(layer_act, dim=(3, 4), keepdim=True)
        print(averaged_tensor.shape)
        pca_components, explained_variance = compute_pca_components_gpu(averaged_tensor.to(device))
        print(pca_components.shape)

        plt.figure(figsize=(8, 4))
        plt.plot(range(1, len(explained_variance.cpu()) + 1), explained_variance.cpu(), marker='o')
        plt.xlabel('Principal Component')
        plt.ylabel('Explained Variance Ratio')
        plt.title('Scree Plot')
        plt.show()

        pca1 = pca_components[0, :].cpu()
        pca2 = pca_components[1, :].cpu()
        plt.figure(figsize=(8, 6))
        plt.scatter(pca1, pca2, alpha=0.5)
        plt.xlabel('PCA1')
        plt.ylabel('PCA2')
        plt.title('Scatter plot of PCA1 vs PCA2')
        plt.grid(True)
        plt.show()

        # Save the components of the spatially averaged activations
        save_pca_path = os.path.join(PCA_DIR, 'pca_components_' + layer_name_pca + '.pt')
        torch.save(pca_components, save_pca_path)
        print('Saved', save_pca_path)

        del layer_act, layer_activations

## 2.2 Load the PCA components
`arr_PCAs[layer]` has shape `[20, 352]`. Row `pca_num - 1` (PC2) becomes the fixed weight vector of that layer's readout classifier (paper Section 2.2.2).

In [ ]:
pca_components_res0 = torch.load(os.path.join(PCA_DIR, 'pca_components_res0.pt'))
pca_components_res1 = torch.load(os.path.join(PCA_DIR, 'pca_components_res1.pt'))
pca_components_res2 = torch.load(os.path.join(PCA_DIR, 'pca_components_res2.pt'))
pca_components_res3 = torch.load(os.path.join(PCA_DIR, 'pca_components_res3.pt'))

arr_PCAs = {
    'res0': pca_components_res0,
    'res1': pca_components_res1,
    'res2': pca_components_res2,
    'res3': pca_components_res3
}

## 2.3 Synthesize the stimuli

For every target layer (`res0`, `res1`, `res2`, `res3`, in this order):

1. **Target pair** (`optim_stim_multiObj`, Section 2.2.3). One random pattern is duplicated into a pair (labels 0 and 1). Each of 2 000 Adam iterations masks the pair, applies random jitter (decaying to zero) and random rescaling, runs DorsalNet, and minimizes the negative BCE of the target layer's fixed PC2 classifier (central spatial unit) plus four penalties that keep the stimuli neutral gray, colour-free and at a fixed contrast.
2. **Optimization levels** (`generate_noise_levels`, Section 2.2.5). A new random pair is pulled toward the target pair's activations for 1 200 iterations and saved every 50 iterations, giving a graded series from noise to target-like.
3. **Post-processing and export** (Section 2.2.4). Smoothing, masking, 2× up-sampling, removal of the first and last frame, contrast/luminance normalization and grayscale conversion; each stimulus is saved as a GIF.
4. **Diagnostics**: loss plots, 3D PCA of the levels with the classifier plane, and a `present.html` page per layer.

Outputs go to `data/generated_stimuli/<layer>_Inhb_False_classifierPCA2__nex_2_var_1/`:

| File | Content |
|---|---|
| `all_stimuli.pt` | `[26, 2, 3, 11, 164, 164]`: index 0 = optimized target pair, 1–25 = optimization levels (noise → target-like) |
| `level_<k>_basic_animation_<i>.gif` | Display-ready stimulus `i` of level `k` (level 0 = target pair) |
| `losses.png`, `this_loss.npy`, `this_loss_noise.npy`, `loss_at_diff_level.txt` | Loss traces |
| `3D_Visualization_videos.png`, `3D_interactive_plot.html` | PCA of the levels (Fig 2B–C) |
| `present.html`, `variables.txt` | Summary page and the settings used |

The global seed is reset to 42 first. All four layers are generated (only res0 and res2 are used in the experiments) so the random-number stream, and therefore the res2 stimuli, match the paper.

In [ ]:
# ------------------------------------------------------------------
# Set the seed for everything
# ------------------------------------------------------------------
# 1. Python built-in random
random.seed(seed)
# 2. NumPy
np.random.seed(seed)
# 3. PyTorch CPU
torch.manual_seed(seed)
# 4. PyTorch GPU
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
# 5. Deterministic cuDNN behaviour
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Make the stimulus_generation modules importable
if STIM_GEN_DIR not in sys.path:
    sys.path.append(STIM_GEN_DIR)

from presentHTML import make_presnetHTML, generate_htmlFile
from threeD_pca import perform_pca, plot_3D_Visualization, plot_3D_Visualization_interactive
from plotting_func import make_directory_path, plot_losses_layers
from post_stim_anal import post_optim_manipulations, show_animations_in_notebook, post_optim_normalization
from generate_x import generate_x
from general_func import get_actv, save_variables_to_file
from generate_noise_levels import generate_noise_levels
from optim_stim_multiObj import optim_stim_multiObj
from Model_Variables import Model_Variables

# All settings (see stimulus_generation/Model_Variables.py)
var = Model_Variables()


for num in range(1, var.diversity_variation + 1):
    for inhibit_otherLayers in var.inhibit_otherLayers_arr:
        for inhibit_top_layers in var.inhibit_top_layers_arr:
            for var.nex_in in var.nex_in_arr:
                for random_seed in var.rnd_seed_ar:
                    all_stimuli_layers = []
                    for i in range(len(var.layers_actv)):
                        # ------------------------------------------------------
                        # Target layer and (optionally) inhibited layers
                        # ------------------------------------------------------
                        var.actv_layers = [var.layers_actv[i]]
                        if inhibit_otherLayers:
                            if inhibit_top_layers:
                                # Inhibit only the layers after the target layer
                                index = var.all_layers.index(var.actv_layers[0])
                                var.inhb_layers = var.all_layers[index + 1:]
                            else:
                                # Inhibit all other layers
                                var.inhb_layers = [layer for layer in var.all_layers if layer not in var.actv_layers]
                        else:
                            # Paper setting: no inhibition
                            var.inhb_layers = []
                            if inhibit_top_layers:
                                continue

                        for j in range(len(var.pca_num_arr)):
                            var.pca_num = var.pca_num_arr[j]

                            coeffs = [1]  # weight of the classifier term in the objective

                            for coeff in coeffs:
                                this_folder = (var.actv_layers[0] + '_Inhb_' + str(inhibit_otherLayers) + '_classifierPCA' + str(var.pca_num) + '_' +
                                               '_nex_' + str(var.nex_in) + '_var_' + str(num))
                                make_directory_path(os.path.join(var.base_folder, var.branch_folder, this_folder))
                                save_variables_to_file(var, os.path.join(var.base_folder, var.branch_folder, this_folder))

                                # ==============================================
                                # 1. Target pair (Section 2.2.3)
                                # ==============================================
                                x, mask, filt, pad, target_contrast, target_mixed_contrast = generate_x(var)
                                x_optim_list_org, this_loss, loss_atSave_org, classifiers, transforms_optimStim = optim_stim_multiObj(model,
                                                                                                                                      outputs,
                                                                                                                                      var,
                                                                                                                                      x,
                                                                                                                                      mask,
                                                                                                                                      filt,
                                                                                                                                      pad,
                                                                                                                                      target_contrast,
                                                                                                                                      target_mixed_contrast,
                                                                                                                                      CEcoeff=coeff,
                                                                                                                                      arr_PCAs=arr_PCAs,
                                                                                                                                      base_folder=var.base_folder,
                                                                                                                                      branch_folder=var.branch_folder,
                                                                                                                                      this_folder=this_folder)

                                # Classifier term of each layer over iterations (Fig 1F-G)
                                plot_losses_layers(np.array(this_loss)[:, 0:4], os.path.join(var.base_folder, var.branch_folder, this_folder))
                                save_path = os.path.join(var.base_folder, var.branch_folder, this_folder)
                                os.makedirs(save_path, exist_ok=True)
                                np.save(os.path.join(save_path, 'this_loss.npy'), np.array(this_loss))

                                # Export the target pair as GIFs (level 0) and keep its
                                # spatially averaged target-layer activation for the PCA plot
                                if var.save_stim_atIter is False:
                                    for generation_level in range(len(x_optim_list_org)):
                                        if var.save_gif == True:
                                            images = post_optim_manipulations(x_optim_list_org[generation_level], filt, pad)

                                            images_norm, mask_final = post_optim_normalization(images.cpu().detach().numpy())
                                            print('imagenorm: ', images_norm.shape)
                                            show_animations_in_notebook(images_norm * mask_final,
                                                                        os.path.join(var.base_folder, var.branch_folder, this_folder),
                                                                        generation_level=generation_level,
                                                                        grayScale=var.grayScale)
                                        activation_layer = get_actv(model, outputs, x_optim_list_org[0].clone().detach(), var.actv_layers[0], mask, transforms_optimStim)
                                        actv_layer_avg = torch.mean(activation_layer, dim=(-2, -1))
                                        x_optim_list_org_actv = []
                                        x_optim_list_org_actv.append(actv_layer_avg.clone().detach())
                                else:
                                    x_optim_list_org_actv = x_optim_list_org

                                # ==============================================
                                # 2. Optimization levels: from full noise toward
                                #    the target pair (Section 2.2.5)
                                # ==============================================
                                if var.gen_noise_case is True:
                                    # Target activations, computed once and held fixed
                                    actv_x = get_actv(model, outputs, x_optim_list_org[0], var.actv_layers[0], mask, transforms_optimStim)
                                    # New random starting pair
                                    x_n, mask_n, filt_n, pad_n, target_contrast_n, target_mixed_contrast_n = generate_x(var)
                                    x_optim_list_noise, this_loss_noise, loss_atSave_noise, full_x_optim_list = generate_noise_levels(model,
                                                                                                                                      transforms_optimStim,
                                                                                                                                      outputs,
                                                                                                                                      var,
                                                                                                                                      actv_x,
                                                                                                                                      x_n,
                                                                                                                                      mask_n,
                                                                                                                                      filt_n,
                                                                                                                                      pad_n,
                                                                                                                                      target_contrast_n,
                                                                                                                                      target_mixed_contrast_n,
                                                                                                                                      base_folder=var.base_folder,
                                                                                                                                      branch_folder=var.branch_folder,
                                                                                                                                      this_folder=this_folder)

                                    save_path = os.path.join(var.base_folder, var.branch_folder, this_folder)
                                    os.makedirs(save_path, exist_ok=True)
                                    np.save(os.path.join(save_path, 'this_loss_noise.npy'), np.array(this_loss_noise))

                                    x_optim_list_merged = x_optim_list_org_actv + x_optim_list_noise
                                    loss_atSave = loss_atSave_org + loss_atSave_noise
                                else:
                                    x_optim_list_merged = x_optim_list_org_actv
                                    loss_atSave = loss_atSave_org

                                # Target pair + all levels -> one tensor [n_levels, 2, 3, 11, 164, 164]
                                if var.gen_noise_case is True:
                                    all_tensors = x_optim_list_org + full_x_optim_list
                                else:
                                    all_tensors = x_optim_list_org
                                all_stimuli = torch.stack(all_tensors, dim=0)
                                all_stimuli_layers.append(all_stimuli)

                                # ==============================================
                                # 3. Summary page, loss values and 3D PCA plots
                                # ==============================================
                                folder_path = os.path.join(var.base_folder, var.branch_folder, this_folder)
                                image_levels = len(loss_atSave)
                                if image_levels == None:
                                    num_levels = 1
                                else:
                                    num_levels = int(image_levels)
                                make_presnetHTML(num_levels, folder_path, loss_atSave, nex=var.nex_in)

                                # Loss value at each saved level
                                loss_atSave_cpu = [tensor.detach().cpu().numpy() for tensor in loss_atSave]
                                output_file = os.path.join(var.base_folder, var.branch_folder, this_folder, 'loss_at_diff_level.txt')
                                with open(output_file, 'w') as file:
                                    for i, tensor in enumerate(loss_atSave_cpu):
                                        file.write(f'{i}: {tensor}\n')

                                # Classifier of the target layer
                                for i in range(len(classifiers)):
                                    if classifiers[i].control_actv == +1:
                                        index_layer = i
                                        break
                                # 3D PCA of the levels with the classifier plane (Fig 2B-C)
                                if var.gen_noise_case is True:
                                    pca_result_tensor, classifier_projection = perform_pca(x_optim_list_merged, var.actv_layers[0], classifier_weights=classifiers[index_layer].classifier_weights[0], save_stim=True)
                                    plot_3D_Visualization(pca_result_tensor, var.nex_in, classifier_projection, classifiers[index_layer].classifier_bias, save_img=True,
                                                          base_folder=var.base_folder, branch_folder=var.branch_folder, this_folder=this_folder, display_classifier=True, classifier=var.loss_func)
                                    plot_3D_Visualization_interactive(pca_result_tensor, var.nex_in, classifier_projection, classifiers[index_layer].classifier_bias, save_img=True,
                                                                      base_folder=var.base_folder, branch_folder=var.branch_folder, this_folder=this_folder, display_classifier=True, classifier=var.loss_func)

                                # Raw stimuli of this layer (read by Section 3)
                                save_path = os.path.join(var.base_folder, var.branch_folder, this_folder, "all_stimuli.pt")
                                torch.save(all_stimuli, save_path)

# Overview page for all layers
generate_htmlFile(os.path.join(var.base_folder, var.branch_folder))
# Stimuli of all layers in one file
save_path = os.path.join(var.base_folder, var.branch_folder, "all_stimuli_layers.pt")
torch.save(all_stimuli_layers, save_path)

---
# 3. ANN readout modeling experiment (paper Section 2.4)

Which readout of the frozen DorsalNet reproduces the human specificity pattern? A linear classifier with a sigmoid output reads the **central spatial unit** of one or more layers (32 channels × 11 frames = 352 features per layer) and learns to classify single stimuli into their category of origin (label 0 / 1). DorsalNet stays frozen; only the classifier weights (initialized to zero) are trained.

| `readout` | Reads from | Hypothesis |
|---|---|---|
| `res0`, `res1` | one early layer | early-stage reweighting |
| `res2`, `res3` | one late layer | late-stage reweighting |
| `modelOutput` | DorsalNet's output | — |
| `All` | res0–res3 concatenated (1 408 features) | learning picks the layer(s) |
| `singleLayer` | res0 for res0 stimuli, res2 for res2 stimuli | stimulus-driven (adaptive-stage) reweighting |

For each readout, one classifier is trained on res0 stimuli and one on res2 stimuli (Adam, lr = 1e-5, 10 000 epochs, the fully optimized target pair as training set, a fresh grayscale noise patch on every pass). After every step each classifier is evaluated on
* the **training** stimuli,
* the **location-transfer** set: the same stimuli shifted toward the bottom-right of the 164 × 164 input,
* the **stimulus-category-transfer** set: the other category's stimuli.

Code: `ann_readout/`. All settings: `ann_readout/beh_Model_Variables.py`. Each readout's results are pickled to `results/ann_readout/run_<timestamp>_epochs_10000/beh_results_readout_<readout>_ablate_False.pkl`.

The global seed is reset to 42 first. The readouts run in the order of `readout_arr`, and the random state carries over from one readout to the next.

In [ ]:
# ------------------------------------------------------------------
# Set the seed for everything
# ------------------------------------------------------------------
# 1. Python built-in random
random.seed(seed)
# 2. NumPy
np.random.seed(seed)
# 3. PyTorch CPU
torch.manual_seed(seed)
# 4. PyTorch GPU
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
# 5. Deterministic cuDNN behaviour
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

import pickle
from datetime import datetime

# Make the ann_readout (and stimulus_generation) modules importable
for p in [STIM_GEN_DIR, ANN_READOUT_DIR]:
    if p not in sys.path:
        sys.path.append(p)

from beh_classifier_model import Classifier_centralpixel_binary
from beh_Model_Variables import beh_Model_Variables
from load_all_gen_imgs import load_all_gen_imgs
from prepare_model_in_out import prepare_model_in_out
from simultaneous_train_test_model import simultaneous_train_test_model_binary
from plotting_tools import LossPlotter

# All settings (see ann_readout/beh_Model_Variables.py)
var = beh_Model_Variables()

if var.repeatNotVar is True:
    enumerate_var = [str(i) for i in np.arange(1, var.repeat + 1)]
else:
    enumerate_var = var.variation_array

# ------------------------------------------------------------------
# One results folder per run
# ------------------------------------------------------------------
base_dir = os.path.join(RESULTS_DIR, 'ann_readout')
os.makedirs(base_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RUN_FOLDER = os.path.join(base_dir, f"run_{timestamp}_epochs_{var.num_epochs}")
os.makedirs(RUN_FOLDER, exist_ok=True)


for readout_index, readout in enumerate(var.readout_arr):
    var.readout = readout
    for ablate_index, ablate in enumerate(var.ablate_arr):
        var.ablate = ablate
        print(f"Starting readout variation: {var.readout}")
        print(f"Starting abalate variation: {var.ablate}")
        for var_index, var.variation in enumerate(enumerate_var):
            # -------------------- Load the synthesized stimuli -------------------- #
            beh_X_res0, beh_Y_res0, beh_X_res2, beh_Y_res2 = load_all_gen_imgs(var, device=device)

            # ------------ Training set and location-transfer set ------------ #
            (xRes0_train,
             yRes0,
             xRes0_loc_test,
             xRes2_train,
             yRes2,
             xRes2_loc_test) = prepare_model_in_out(var,
                                                    device,
                                                    beh_X_res0,
                                                    beh_Y_res0,
                                                    beh_X_res2,
                                                    beh_Y_res2)

            # ------------- Classifiers: one per stimulus category ------------- #
            spatial_size = xRes0_train.shape[-1]
            print(f"Spatial size of input images: {spatial_size}x{spatial_size}")

            classifier_singleLayerReadout_res0 = Classifier_centralpixel_binary(model, classifier_w='zero', dropout=var.dropout,
                                                                                spatial_size=spatial_size, readout=var.readout).to(device)
            classifier_singleLayerReadout_res2 = Classifier_centralpixel_binary(model, classifier_w='zero', dropout=var.dropout,
                                                                                spatial_size=spatial_size, readout=var.readout).to(device)

            # ----------------------- Train and test ----------------------- #
            results = simultaneous_train_test_model_binary(
                var=var,
                device=device,
                outputs=outputs,
                # ------------ res0 stimuli --------------
                xRes0_train=xRes0_train,
                yRes0=yRes0,
                xRes0_loc_test=xRes0_loc_test,
                # ------------ res2 stimuli --------------
                xRes2_train=xRes2_train,
                yRes2=yRes2,
                xRes2_loc_test=xRes2_loc_test,
                # ------------ classifiers ---------------
                classifier_singleLayerReadout_res0=classifier_singleLayerReadout_res0,
                classifier_singleLayerReadout_res2=classifier_singleLayerReadout_res2
            )

            # --------------------------- Plots --------------------------- #
            plotter = LossPlotter(results)
            plotter.plot_binary_summary_performance_confidence(clf_key=var.readout)
            plotter.plot_binary_specificity(clf_key=var.readout)

            # ----------------------- Save results ----------------------- #
            file_path = os.path.join(
                RUN_FOLDER,
                f"beh_results_readout_{var.readout}_ablate_{var.ablate}.pkl"
            )
            with open(file_path, "wb") as f:
                pickle.dump(results, f)

            print(f"Saved results to:\n{file_path}")

---
# 4. Visualization

* **4.1** Synthesized stimuli (Section 2 output).
* **4.2** Training and location-test performance of one readout (Section 3 output).
* **4.3** Specificity indices of every readout (Fig 7).

Performance of a classifier is 1 − |prediction − label|. The specificity index computed below is

$$SI = \mathrm{clip}\left(\frac{P_\mathrm{train} - P_\mathrm{test}}{P_\mathrm{train} - P_\mathrm{train}^{\,\mathrm{epoch}\,0}},\ -1,\ 1\right)$$

i.e. the performance lost on the transfer set relative to the improvement achieved during training (0 = full transfer, 1 = fully specific).

## 4.1 Synthesized stimuli

In [ ]:
from IPython.display import Image, display

plt.rcdefaults()  # start from Matplotlib defaults (LossPlotter changes the global style)

GEN_ROOT = os.path.join(REPO_ROOT, 'data', 'generated_stimuli')

for layer in ['res0', 'res2']:
    folder = os.path.join(GEN_ROOT, f'{layer}_Inhb_False_classifierPCA2__nex_2_var_1')
    print(f'{layer}: fully optimized pair (level 0)')
    for i in range(2):
        display(Image(filename=os.path.join(folder, f'level_0_basic_animation_{i}.gif')))
    display(Image(filename=os.path.join(folder, '3D_Visualization_videos.png')))
    print('All levels of this layer:', os.path.join(folder, 'present.html'))

print('Overview of all layers:', os.path.join(GEN_ROOT, 'present.html'))

## 4.2 Training and testing performance of one readout
Set `DATA_DIR` to a run folder from Section 3 (by default the run just completed) and `readout` to the architecture to plot.

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

def avg_all(arr):
    '''[epochs, n_stimuli, 1] -> mean over stimuli -> [epochs]'''
    arr = arr.squeeze(-1)
    return arr.mean(axis=1)

# Results folder of the ANN run
DATA_DIR = RUN_FOLDER if 'RUN_FOLDER' in globals() else os.path.join(RESULTS_DIR, 'ann_readout', '<run folder>')

readouts = ['res0', 'res1', 'res2', 'res3', 'modelOutput', 'All']

readout = 'res3'
filename = f"beh_results_readout_{readout}_ablate_False.pkl"
filepath = os.path.join(DATA_DIR, filename)

with open(filepath, "rb") as f:
    results = pickle.load(f)

# Performance (1 - loss) for res0 and res2 stimuli
res0_train = 1 - avg_all(np.array(results["res0"][readout]["loss_train"]))
res2_train = 1 - avg_all(np.array(results["res2"][readout]["loss_train"]))

res0_loc  = 1 - avg_all(np.array(results["res0"][readout]["loss_loc"]))
res2_loc  = 1 - avg_all(np.array(results["res2"][readout]["loss_loc"]))
res0_stim = 1 - avg_all(np.array(results["res0"][readout]["loss_stim"]))
res2_stim = 1 - avg_all(np.array(results["res2"][readout]["loss_stim"]))

# Specificity indices
si_res0_loc = np.clip((res0_train - res0_loc) / (res0_train - res0_train[0] + 1e-8), -1, 1.0)
si_res2_loc = np.clip((res2_train - res2_loc) / (res2_train - res2_train[0] + 1e-8), -1, 1.0)

si_res0_stim = np.clip((res0_train - res0_stim) / (res0_train - res0_train[0] + 1e-8), -1, 1.0)
si_res2_stim = np.clip((res2_train - res2_stim) / (res2_train - res2_train[0] + 1e-8), -1, 1.0)

In [ ]:
# ------------------------------------------------------------------
# Drawn at the final printed size (inches) so fonts stay readable.
# Change FIG_W to the width this subplot takes on the page.
# ------------------------------------------------------------------
FIG_W, FIG_H = 3.3, 2.6

plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12,
    'lines.linewidth': 2.6,
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.major.pad': 2,
    'ytick.major.pad': 2,
    'xtick.direction': 'out',    # ticks point away from the data
    'ytick.direction': 'out',
    'savefig.dpi': 600,
    'pdf.fonttype': 42,          # keeps text editable in Illustrator/Inkscape
})

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

n_train = len(res0_train)
epochs = np.arange(n_train)
test_x = n_train + 300           # test point just after training

# Training curves
ax.plot(epochs, res0_train, color='#c2a5cf', alpha=0.8, label='res0 train')
ax.plot(epochs, res2_train, color='#a6dba0', alpha=0.8, label='res2 train')

# Final location-test points
ax.scatter(test_x, res0_loc[-1], color='#7b3294', s=70, zorder=5, label='res0 test')
ax.scatter(test_x, res2_loc[-1], color='#008837', s=70, zorder=5, label='res2 test')

ax.set_xlabel('Epoch', labelpad=2)
ax.set_ylabel('Performance', labelpad=2)

# Y axis: 0.5, 0.75, 1 (extra room on top for the Training/Testing labels)
ax.set_ylim(0.48, 1.09)
ax.set_yticks([0.5, 0.75, 1])
ax.set_yticklabels(['0.5', '0.75', '1'])

# X axis: 3 ticks, written short
ax.set_xlim(-200, test_x + 600)
ax.set_xticks([0, n_train // 2, n_train])
ax.set_xticklabels(['0', f'{n_train // 2 // 1000}k', f'{n_train // 1000}k'])

# Only left and bottom spines, offset from the data
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_position(('outward', 6))

# Axis lines stop at the first and last tick
ax.spines['left'].set_bounds(0.5, 1)
ax.spines['bottom'].set_bounds(0, n_train)

# Phase labels
ax.text(n_train * 0.45, 1.05, 'Training', fontsize=13, fontweight='bold', ha='center')
ax.text(test_x, 1.05, 'Testing', fontsize=13, fontweight='bold', ha='center')

ax.legend(frameon=False, loc='lower center', ncol=2,
          handlelength=1.0, handletextpad=0.4, columnspacing=0.8)

fig.tight_layout(pad=0.3)

fig.savefig(
    os.path.join(FIGURE_DIR, f"TrainingPerformance_{readout}.pdf"),
    bbox_inches="tight", pad_inches=0.02, transparent=True
)
plt.show()

## 4.3 Specificity index of every readout (Fig 7)
Location and stimulus-category specificity at the last epoch, for res0- and res2-trained classifiers. Set `SAVE_FIGS = True` to save each figure to `results/figures/`.

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Global style: Times New Roman, bold, large, ticks outward
# =========================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.bf': 'Times New Roman:bold',
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,   # keep text editable in Illustrator / Inkscape
    'ps.fonttype': 42,
})

# Font sizes (large so they stay readable when the figure is scaled down)
LABEL_FS = 32
TICK_FS = 28
TITLE_FS = 32

# Line weights
SPINE_LW = 3.5
TICK_W = 3.5
TICK_LEN = 10
BAR_EDGE_LW = 2.5

# Bars
BAR_WIDTH = 0.35
X_POSITIONS = [0, 0.55]
COLORS = ['#AF8DC3', '#A6DBA0']   # res0, res2

SAVE_FIGS = False                 # set True to save each figure
SAVE_FORMATS = ['pdf', 'png']

test_epoch = 9999  # last epoch (10 000 epochs)


def avg_all(arr):
    arr = arr.squeeze(-1)
    return arr.mean(axis=1)


# Results folder of the ANN run
DATA_DIR = RUN_FOLDER if 'RUN_FOLDER' in globals() else os.path.join(RESULTS_DIR, 'ann_readout', '<run folder>')

readouts = ['res0', 'res1', 'res2', 'res3', 'modelOutput', 'All', 'singleLayer']

# Figure title for each readout
READOUT_TITLES = {
    'res0': 'Res0 readout',
    'res1': 'Res1 readout',
    'res2': 'Res2 readout',
    'res3': 'Res3 readout',
    'modelOutput': 'Model output readout',
    'All': 'All layers readout',
    'singleLayer': 'Adaptive-stage readout',
}


def style_axis(ax, show_ylabel):
    '''Publication styling for one subplot.'''
    # x axis
    ax.set_xticks(X_POSITIONS)
    ax.set_xticklabels(['res0', 'res2'], fontsize=LABEL_FS, fontweight='bold')
    ax.set_xlim(X_POSITIONS[0] - 0.4, X_POSITIONS[1] + 0.4)

    # y axis: 0, 0.5, 1
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([0, 0.5, 1])
    ax.set_yticklabels(['0', '0.5', '1'], fontsize=TICK_FS, fontweight='bold')
    if show_ylabel:
        ax.set_ylabel('Specificity', fontsize=LABEL_FS, fontweight='bold')

    # thick ticks pointing outward
    ax.tick_params(axis='both', which='major', direction='out',
                   width=TICK_W, length=TICK_LEN, pad=6)
    ax.tick_params(axis='x', length=0)   # no tick marks under categorical bars

    # hide top/right spines, thicken left/bottom
    for name, spine in ax.spines.items():
        if name in ['top', 'right']:
            spine.set_visible(False)
        else:
            spine.set_linewidth(SPINE_LW)
    ax.spines['left'].set_bounds(0, 1)   # y spine runs exactly from 0 to 1


for readout in readouts:
    filename = f"beh_results_readout_{readout}_ablate_False.pkl"
    filepath = os.path.join(DATA_DIR, filename)

    with open(filepath, "rb") as f:
        results = pickle.load(f)

    # Performance (1 - loss) for res0 and res2 stimuli
    res0_train = 1 - avg_all(np.array(results["res0"][readout]["loss_train"]))
    res2_train = 1 - avg_all(np.array(results["res2"][readout]["loss_train"]))

    res0_loc  = 1 - avg_all(np.array(results["res0"][readout]["loss_loc"]))
    res2_loc  = 1 - avg_all(np.array(results["res2"][readout]["loss_loc"]))
    res0_stim = 1 - avg_all(np.array(results["res0"][readout]["loss_stim"]))
    res2_stim = 1 - avg_all(np.array(results["res2"][readout]["loss_stim"]))

    # Specificity indices
    si_res0_loc = np.clip((res0_train - res0_loc) / (res0_train - res0_train[0] + 1e-8), -1, 1.0)
    si_res2_loc = np.clip((res2_train - res2_loc) / (res2_train - res2_train[0] + 1e-8), -1, 1.0)

    si_res0_stim = np.clip((res0_train - res0_stim) / (res0_train - res0_train[0] + 1e-8), -1, 1.0)
    si_res2_stim = np.clip((res2_train - res2_stim) / (res2_train - res2_train[0] + 1e-8), -1, 1.0)

    # Two subplots: location and stimulus specificity
    fig, axes = plt.subplots(1, 2, figsize=(7, 5.5), sharey=True)
    fig.suptitle(READOUT_TITLES.get(readout, f'{readout} readout'),
                 fontsize=TITLE_FS + 2, fontweight='bold')

    panels = [
        ('Location', si_res0_loc, si_res2_loc),
        ('Stimulus', si_res0_stim, si_res2_stim),
    ]

    for i, (title, si_res0, si_res2) in enumerate(panels):
        ax = axes[i]
        values = [si_res0[test_epoch], si_res2[test_epoch]]

        ax.bar(X_POSITIONS, values, width=BAR_WIDTH, color=COLORS,
               edgecolor='black', linewidth=BAR_EDGE_LW)

        ax.set_title(title, fontsize=TITLE_FS, fontweight='bold')
        style_axis(ax, show_ylabel=(i == 0))

    plt.tight_layout(rect=[0, 0, 1, 0.93])   # room for the title

    if SAVE_FIGS:
        for ext in SAVE_FORMATS:
            fig.savefig(os.path.join(FIGURE_DIR, f"specificity_{readout}_epoch{test_epoch}.{ext}"),
                        dpi=300, bbox_inches='tight')

    plt.show()